# Lesson 1 : Create a basic agent

In this exercise, we learn how to build and run agent in Agent Framework.

Agent Framework supports various backend clients - such as, OpenAI Assistants API, OpenAI Responses API, Anthropic Client, Ollama, Azure AI agents, etc. (For other clients, you can easily build your own custom agent for Agent Framework from scratch.)  
Throughout this workshop, we use a client which connects to Microsoft Foundry (which is based on ```azure-ai-projects``` version 2 SDK).

> Note : In ```azure-ai-projects``` v2 SDK, Azure OpenAI Responses API or Conversations are transparently called internally.

Before starting, please remember to perform the preparations described in [Readme.md](./Readme.md).

Now let's start creating a custom agent.

Firstly, we create a client object to connect to Azure AI as follows.  
All the difference depending on individual clients is handled on this client's object.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(credential=credential)

Next, create an agent using above client.  
In this example, we'll use local tool calling in the agent.

In [2]:
# define local tools
from typing import Annotated
from pydantic import Field
from random import randint

def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],  # "天気を取得する場所"
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]  # "晴れ", "曇り", "雨", "嵐"
    return f"The weather in {location} is {conditions[randint(0, 3)]}."  # "{location}の天気は{conditions[randint(0, 3)]}です。"

def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],  # "気温を取得する場所"
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."  # "{location}の気温は{randint(10, 30)}°Cです。"

In [3]:
from agent_framework import ChatAgent

# create agent
agent = ChatAgent(
    name="BasicWeatherAgent",
    chat_client=client,
    instructions="You are an agent about weather information.",  # "あなたは、気象情報に関する Agent です。"
    tools=[get_weather, get_temperature])

Let's run the agent.  
You'll see that the function tools are being used appropriately.

> Note : Because we're using ```azure-ai-projects``` client, the agent is registered in Microsoft Foundry by running this call.

In [4]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka.")  # "大阪の天気と気温を教えてください。"
display(Markdown(result.text))

Osaka is **cloudy**, and the temperature is **22 °C**.

Let's see the list of internal messages.  
You'll see that ```get_weather()``` and ```get_temperature()``` are concurrently called by the function calling.

In [5]:
import agent_framework

for i, msg in enumerate(result.messages):
    print(f"********** message {i} **********")
    for c in msg.contents:
        if c.type == "function_call":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function name : {c.name}")
            print(f"function arguments : {c.arguments}")
        elif c.type == "function_result":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function result : {c.result}")
            print(f"exceptions : {c.exception}")
        elif c.type == "text":
            print(f"*** {c.type} ***")
            print(f"text : {c.text}")
        else:
            print(f"*** Other types : {c.type}***")

********** message 0 **********
*** function_call ***
call id : call_3JWiYmxuafK82rtIlMUhYd3J
function name : get_weather
function arguments : {"location":"Osaka"}
*** function_call ***
call id : call_UGV7t0Be9fWRsm2RFXwq2iyz
function name : get_temperature
function arguments : {"location":"Osaka"}
********** message 1 **********
*** function_result ***
call id : call_3JWiYmxuafK82rtIlMUhYd3J
function result : The weather in Osaka is cloudy.
exceptions : None
*** function_result ***
call id : call_UGV7t0Be9fWRsm2RFXwq2iyz
function result : The temperature in Osaka is 22 degrees.
exceptions : None
********** message 2 **********
*** text ***
text : Osaka is **cloudy**, and the temperature is **22 °C**.


For the fluent user experience, you can also perform streaming outputs as follows. (Each character will be displayed one by one.)

In [6]:
async def streaming_example():
    async for chunk in agent.run_stream("Tell me the weather and temperature in Osaka."):  # "大阪の天気と気温を教えてください。"
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print("\n")

await streaming_example()

Osaka weather: rainy.  
Osaka temperature: 14 °C.

